# Lab 7.5 &mdash; Challenge: The Release Gate

**Level:** Advanced &middot; challenge &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 3 &middot; Module 7 &mdash; Multi-Agent System Evaluation**

### What you'll do
- Write the acceptance bar down as a typed object, before you look at a result
- Choose which quality test the gate uses &mdash; and find what the other one lets through
- Decide whether a disqualifying behaviour is a reason or the end of the discussion
- Put one number in front of the person who owns the budget
- Produce a ship / do-not-ship decision with reasons for four real candidates

> **How this lab works.** You write real LangChain code &mdash; the agent under test, the callback
> handler that traces it, the typed verdict you grade. Fill every `BLANK`, then run the
> **Self-check** cell under each section. Those check the *objects you built* and the *recorded
> runs* shipped in the notebook, so they are deterministic and do not depend on the model.
> Cells marked **Run it for real** put your code in front of the sandbox model; that is the part
> worth watching, and it is never scored &mdash; scoring a live run would contradict Lab 7.1.

> **The deliverable of Day 3.** Four candidates, a bar agreed in advance, and one
> decision per candidate that a release manager could act on this afternoon.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap, random, statistics
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-7-05")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Off is the default here because an eval lab makes a lot of calls.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- four candidate versions, already measured
# Each was run 30 times over a 50-case eval set -- the shape Lab 7.1 showed you need before a
# difference is even expressible. Cost is per case; latency is p95 seconds.

CANDIDATES = {
    "v6 (current)": {
        "rates": [0.76, 0.76, 0.66, 0.74, 0.66, 0.76, 0.74, 0.86, 0.74, 0.78, 0.74, 0.74, 0.72,
                  0.8, 0.78, 0.7, 0.76, 0.66, 0.88, 0.88, 0.8, 0.74, 0.62, 0.84, 0.66, 0.8, 0.7,
                  0.64, 0.74, 0.82],
        "cost_per_case": 0.0121, "p95_latency_s": 11.4},
    "v7 better prompt": {
        "rates": [0.82, 0.74, 0.84, 0.8, 0.82, 0.64, 0.84, 0.88, 0.86, 0.86, 0.82, 0.8, 0.8, 0.86,
                  0.88, 0.74, 0.8, 0.68, 0.78, 0.94, 0.74, 0.74, 0.78, 0.82, 0.86, 0.86, 0.82,
                  0.86, 0.86, 0.82],
        "cost_per_case": 0.0129, "p95_latency_s": 11.9},
    "v8 more agents": {
        "rates": [0.98, 0.94, 0.98, 0.94, 0.96, 0.92, 0.94, 0.96, 0.94, 0.98, 0.92, 0.98, 0.96,
                  0.96, 0.94, 0.96, 0.98, 0.96, 0.9, 0.96, 1.0, 0.96, 0.96, 0.94, 0.96, 0.98,
                  0.92, 0.94, 0.96, 0.98],
        "cost_per_case": 0.0402, "p95_latency_s": 19.6},
    "v9 cheaper model": {
        "rates": [0.6, 0.68, 0.54, 0.6, 0.56, 0.64, 0.72, 0.62, 0.66, 0.58, 0.48, 0.54, 0.58,
                  0.52, 0.5, 0.52, 0.6, 0.54, 0.64, 0.44, 0.7, 0.42, 0.66, 0.54, 0.64, 0.46,
                  0.56, 0.58, 0.56, 0.62],
        "cost_per_case": 0.0058, "p95_latency_s": 8.2},
}

CURRENT = "v6 (current)"
CASES_PER_DAY = 20000

# Carried forward from Lab 7.3: the worst trajectory verdict each candidate produced on the
# adversarial cases. Everything else about these candidates is a number. This one is a
# behaviour, and Lab 7.3 decided it does not trade off against a pass rate.
TRAJECTORY = {
    "v6 (current)":     "PASS",
    "v7 better prompt": "REJECTED",   # one run attempted release_payment on a sanctions hold
    "v8 more agents":   "PASS",
    "v9 cheaper model": "PASS",
}

print(f"{len(CANDIDATES)} candidates, 30 runs each")

## Concept

A dashboard reports. A gate decides. The difference is whether the build fails.

Three thresholds, agreed while nobody was under pressure &mdash; quality must not regress, cost per
case must stay under a ceiling, p95 latency must stay under a ceiling &mdash; plus one thing that is
not a threshold at all: a behaviour that disqualifies a candidate however good its numbers are.

The interesting candidates are the ones that pass three and fail one.

## Section 1 &mdash; The acceptance bar, written down first

A typed object, not three loose floats, because a gate is read by machines (a CI job) and by
people (whoever it just blocked) and both need to see the same numbers.

In [ ]:
from pydantic import BaseModel, Field

class Thresholds(BaseModel):
    """The acceptance bar. Agreed in a calm week, before anybody had a release to push."""
    cost_ceiling: float = Field(description="Maximum cost per case, in dollars")
    p95_ceiling: float = Field(description="Maximum p95 latency, in seconds")
    quality_tolerance: float = Field(
        description="How far the mean pass rate may fall below the current version and "
                    "still count as no regression")

BAR = Thresholds(cost_ceiling=0.020, p95_ceiling=15.0, quality_tolerance=0.02)


def summarise(name: str) -> dict:
    """A mean and a range, because Lab 7.1 established that a bare number is not reportable."""
    rates = CANDIDATES[name]["rates"]
    return {"name": name, "mean": round(statistics.mean(rates), 4),
            "low": min(rates), "high": max(rates),
            "cost": CANDIDATES[name]["cost_per_case"],
            "p95": CANDIDATES[name]["p95_latency_s"]}


def ranges_overlap(a: str, b: str) -> bool:
    ra, rb = CANDIDATES[a]["rates"], CANDIDATES[b]["rates"]
    return not (min(rb) > max(ra) or min(ra) > max(rb))

In [ ]:
# --- Self-check: Section 1   (a Pydantic bar over recorded measurements -- no model call)
check("the bar is a validated object with all three thresholds",
      lambda: set(Thresholds.model_fields) == {"cost_ceiling", "p95_ceiling",
                                               "quality_tolerance"})
check("every threshold carries a description",
      lambda: all(f.description for f in Thresholds.model_fields.values()),
      "a number with no units is how two people agree to different bars")
check("the current version averages about 75%",
      lambda: 0.74 < summarise(CURRENT)["mean"] < 0.76)
check("v8 is the strongest on quality",
      lambda: max(CANDIDATES, key=lambda n: summarise(n)["mean"]) == "v8 more agents")
check("v9 is the weakest",
      lambda: min(CANDIDATES, key=lambda n: summarise(n)["mean"]) == "v9 cheaper model")
check("only v8 is PROVABLY different from the current version",
      lambda: [n for n in CANDIDATES if n != CURRENT and not ranges_overlap(CURRENT, n)]
              == ["v8 more agents"],
      "v7 is better on average and its range still overlaps v6's -- not proven, on 30 runs")
check("v9's range overlaps the current version's too",
      lambda: ranges_overlap(CURRENT, "v9 cheaper model") is True,
      "so a test that only blocks PROVEN regressions would let v9 straight through")

def _summary():
    print(f"  {'candidate':20}{'mean':>8}{'range':>14}{'cost':>9}{'p95':>7}{'trajectory':>12}")
    print("  " + "-" * 72)
    for n in CANDIDATES:
        s = summarise(n)
        print(f"  {n:20}{s['mean']:>8.1%}{s['low']:>7.0%}-{s['high']:<6.0%}"
              f"{s['cost']:>9.4f}{s['p95']:>7.1f}{TRAJECTORY[n]:>12}")
guard(_summary)

## Section 2 &mdash; The gate

Two decisions here, and neither is arithmetic.

The first is which quality test to use. Lab 7.1 refused to call a difference real while two ranges
overlapped &mdash; that was the right rule for a claim. A gate is not a claim: it is a default, and
somebody lives with whichever way the doubt falls.

The second is what to do with a run that tried something it must never try.

In [ ]:
class GateDecision(BaseModel):
    """One candidate's decision, in a shape a CI job can act on."""
    candidate: str
    ship: bool
    rejected: bool = Field(description="True if a disqualifying behaviour was observed")
    reasons: list[str] = Field(description="Every reason against shipping, in plain words")


def quality_regressed(name: str, current: str = CURRENT) -> bool:
    """Has quality regressed enough to block the release?"""
    mean_rule = (summarise(current)["mean"] - summarise(name)["mean"]) > BAR.quality_tolerance
    proven_rule = (not ranges_overlap(current, name)
                   and summarise(name)["mean"] < summarise(current)["mean"])
    return mean_rule


def decide(name: str) -> GateDecision:
    """Should this version ship? A gate that says only "blocked" gets overridden; one that
    says why gets fixed."""
    s = summarise(name)
    reasons = []
    if quality_regressed(name):
        reasons.append(f"quality {s['mean']:.1%} is more than {BAR.quality_tolerance:.0%} "
                       f"below the current {summarise(CURRENT)['mean']:.1%}")
    if s["cost"] > BAR.cost_ceiling:
        reasons.append(f"cost {s['cost']:.4f} per case exceeds the ceiling "
                       f"{BAR.cost_ceiling:.4f}")
    if s["p95"] > BAR.p95_ceiling:
        reasons.append(f"p95 latency {s['p95']:.1f}s exceeds the ceiling "
                       f"{BAR.p95_ceiling:.1f}s")

    rejected = TRAJECTORY[name] == "REJECTED"

    if rejected:
        reasons.append("a run attempted a forbidden action -- disqualifying, "
                       "independently of every number above")
    return GateDecision(candidate=name, ship=not reasons and not rejected,
                        rejected=rejected, reasons=reasons)

In [ ]:
# --- Self-check: Section 2
def passes_numbers(name: str) -> bool:
    """Would this candidate have shipped on the three thresholds alone?"""
    s = summarise(name)
    return (not quality_regressed(name) and s["cost"] <= BAR.cost_ceiling
            and s["p95"] <= BAR.p95_ceiling)

check("the decision is a validated object, not a tuple",
      lambda: isinstance(decide(CURRENT), GateDecision))
check("the current version passes its own gate",
      lambda: decide(CURRENT).ship is True,
      "a gate the incumbent fails is a gate nobody will agree to")
check("v9 is blocked on quality",
      lambda: decide("v9 cheaper model").ship is False
              and "quality" in decide("v9 cheaper model").reasons[0])
check("THE PROVEN-REGRESSION RULE WOULD HAVE LET V9 THROUGH",
      lambda: ranges_overlap(CURRENT, "v9 cheaper model") is True,
      "a 17-point drop whose range still overlaps: conservative in both directions, and one "
      "of those directions has a user on the end of it")
check("v8 is blocked, and it is the best version on quality",
      lambda: decide("v8 more agents").ship is False,
      "provably better, 3.3x the cost and over the latency ceiling -- the gate does its job")
check("...for two separate reasons, and it is NOT rejected",
      lambda: len(decide("v8 more agents").reasons) == 2
              and decide("v8 more agents").rejected is False,
      "blocked is a budget conversation; rejected is not a conversation")
check("V7 PASSES ALL THREE THRESHOLDS AND STILL DOES NOT SHIP",
      lambda: passes_numbers("v7 better prompt") is True
              and decide("v7 better prompt").ship is False
              and decide("v7 better prompt").rejected is True,
      "the best-looking candidate on the numbers tried to release a sanctioned payment once")
check("nothing ships this week",
      lambda: [n for n in CANDIDATES if decide(n).ship] == [CURRENT])
check("every blocked candidate says why",
      lambda: all(decide(n).reasons for n in CANDIDATES if not decide(n).ship))

def _decisions():
    for n in CANDIDATES:
        g = decide(n)
        print(f"  {'SHIP  ' if g.ship else ('REJECT' if g.rejected else 'BLOCK ')} {n}")
        for r in g.reasons:
            print(f"           - {r}")
guard(_decisions)

## Section 3 &mdash; The number that goes in front of the budget owner

v8 is provably better and blocked on cost. That is not the gate's decision to reverse; it is the
gate's job to hand it to someone who is entitled to make it &mdash; in the units that person decides in.

In [ ]:
def escalation_number(name: str) -> float:
    """One number goes in front of the person who owns the budget. Which one?"""
    per_case = CANDIDATES[name]["cost_per_case"]
    per_day = per_case * CASES_PER_DAY
    return round(per_day, 2)


def escalation(name: str = "v8 more agents") -> dict:
    """The trade, stated so that somebody can say yes or no to it."""
    return {"candidate": name,
            "provably_better": not ranges_overlap(CURRENT, name),
            "blocked_by": decide(name).reasons,
            "extra_per_day": round(escalation_number(name) - escalation_number(CURRENT), 2)}

In [ ]:
# --- Self-check: Section 3
check("THE ESCALATION NUMBER IS IN THE UNITS THE BUDGET IS HELD IN",
      lambda: escalation_number("v8 more agents") > 1.0,
      "0.0402 per case is true and means nothing to anyone who signs for spend")
check("v8 at volume is over eight hundred a day",
      lambda: abs(escalation_number("v8 more agents") - 804.0) < 0.01)
check("the trade is a difference, not a total",
      lambda: escalation()["extra_per_day"] > 500)
check("what is being escalated is genuinely better",
      lambda: escalation()["provably_better"] is True,
      "escalating a version you have not shown is better is how a gate loses its authority")
check("and the escalation says exactly what would have to change",
      lambda: any("cost" in r for r in escalation()["blocked_by"]))
check("the cheapest candidate is not the answer either",
      lambda: decide("v9 cheaper model").ship is False,
      "v9 saves 126 dollars a day and loses 17 points of quality")

def _risks():
    print("  daily cost at 20,000 cases:")
    for n in CANDIDATES:
        print(f"    {n:20} {escalation_number(n):>10.2f}")
    e = escalation()
    print(f"\n  {e['candidate']} is provably better and is blocked.")
    print(f"  Shipping it anyway costs {e['extra_per_day']:.2f} more per day.")
    print("  That is now a decision for whoever owns the budget -- which is the point.")
guard(_risks)

## Run it for real

Write the release note the gate implies. This is what a gate is *for*: not to stop people, but to
make the decision explicit and attributable.

In [ ]:
if llm_ready():
    def _release_note():
        rows = "\n".join(
            f"- {n}: mean {summarise(n)['mean']:.1%} "
            f"(range {summarise(n)['low']:.0%}-{summarise(n)['high']:.0%}), "
            f"cost {summarise(n)['cost']:.4f}/case, p95 {summarise(n)['p95']:.1f}s, "
            f"gate={'SHIP' if decide(n).ship else ('REJECTED' if decide(n).rejected else 'BLOCK')}"
            for n in CANDIDATES)
        reply = ask(
            "Write a short release recommendation for an engineering manager. Say which "
            "version ships, which are blocked and why, which is disqualified, and what "
            "single decision is being escalated.\n\n"
            f"Bar agreed in advance: cost {BAR.cost_ceiling}/case, "
            f"p95 {BAR.p95_ceiling}s, quality tolerance {BAR.quality_tolerance}.\n"
            f"At {CASES_PER_DAY:,} cases a day.\n"
            f"Candidates:\n{rows}")
        print(reply.strip()[:800])
    guard(_release_note)

### Read it

The note should say four things: nothing new ships this week; v7 is one refusal clause away from
shipping and is not a threshold problem; v8 is blocked on cost and latency rather than on quality;
and somebody with a budget needs to decide whether v8's quality is worth about eight hundred
dollars a day.

Notice what the gate did **not** do: it did not decide that. It made the trade explicit, attached
numbers to both sides, and put it in front of a person &mdash; which is the same shape as the approval
gate in Module 5 and the refusal in Module 6. A good control does not remove the judgement. It
makes sure the judgement is made by someone entitled to make it, before the fact rather than after.

And notice which candidate the gate was hardest on. v7 was the best-looking version on every
number the gate measures. One run out of fifty tried something it must never try, and that outranks
every number, because a threshold is a preference and a disqualification is not.

**What you take from Module 7:** one run is a sample; assert on the trajectory as well as the
outcome; keep a span tree, because attribution needs the nesting; diagnose with an ordered ladder;
and put the numbers in a gate rather than on a dashboard.

In [ ]:
score()

## Your turn

1. `quality_tolerance` is 2%. Given the ranges in Section 1, is that inside the noise? Pick a
   defensible value and write the sentence you would use to justify it to the person it blocks.
2. Every gate needs an override path or it gets routed around. Write it down: who can override,
   what they must record, and what happens on the next release. Then decide whether `rejected`
   is overridable at all &mdash; and say why in one line.
3. The gate reads `TRAJECTORY` as a single verdict per candidate. Replace it with the per-run
   verdicts from Lab 7.3 and decide what fraction of rejected runs is still a rejection. If your
   answer is &ldquo;any&rdquo;, check it against Lab 7.1: on fifty cases, is one rejected run distinguishable
   from zero?